# Manga Speaker — Training Notebook (Kaggle)

## Before running this notebook, check:
- ✅ **Accelerator**: Settings (right panel) → Accelerator → **GPU T4 x2** or **P100**
- ✅ **Internet**: Settings → Internet → **On** (needed to download Qwen model from HuggingFace)
- ✅ Both datasets are added to this notebook (see Cell 1 for instructions)

Run every cell **top to bottom**. Training takes ~3-4 hours.

In [ ]:
# ── Cell 1: Check GPU + list input datasets ────────────────────────────────────
# Before running: make sure you added both datasets in the right panel:
#   1. Your project code dataset (manga-speaker-code)
#   2. Your manga images dataset (manga-speaker-data)

!nvidia-smi
print()
print('Input datasets available:')
!ls /kaggle/input/

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# NOTE: Kaggle has many packages pre-installed. We only need a few extras.

!pip install -q \
    transformers>=4.49 \
    peft>=0.10 \
    bitsandbytes>=0.43 \
    qwen-vl-utils \
    langgraph \
    pyyaml

print('Done')

In [ ]:
# ── Cell 3: Extract project code ──────────────────────────────────────────────
# Kaggle input is READ-ONLY, so we copy to /kaggle/working/ first

import os, shutil, glob

WORK = '/kaggle/working/manga_speaker'
os.makedirs(WORK, exist_ok=True)

# Find the project zip in any input dataset
zips = glob.glob('/kaggle/input/**/manga_speaker.zip', recursive=True)
if not zips:
    # Try finding the zip by any name in the code dataset
    zips = glob.glob('/kaggle/input/manga-speaker-code/*.zip')

if zips:
    print(f'Found project zip: {zips[0]}')
    !unzip -q "{zips[0]}" -d /kaggle/working/
    # Handle nested folder from zip
    extracted = glob.glob('/kaggle/working/manga_speaker*', recursive=False)
    print(f'Extracted to: {extracted}')
else:
    print('ERROR: manga_speaker.zip not found in /kaggle/input/')
    print('Add your manga-speaker-code dataset to this notebook first!')
    print('Available inputs:')
    !ls /kaggle/input/

# Set working directory to project root
if os.path.exists('/kaggle/working/manga_speaker'):
    os.chdir('/kaggle/working/manga_speaker')
elif os.path.exists('/kaggle/working/All project/manga_speaker'):
    os.chdir('/kaggle/working/All project/manga_speaker')

print('Working directory:', os.getcwd())
!ls -la

In [ ]:
# ── Cell 4: Extract manga dataset ─────────────────────────────────────────────
import glob, os

DATASET_DIR = os.path.join(os.getcwd(), 'dataset')
os.makedirs(DATASET_DIR, exist_ok=True)

# Find the data zip
data_zips = glob.glob('/kaggle/input/**/BYTE*.zip', recursive=True)
if not data_zips:
    data_zips = glob.glob('/kaggle/input/manga-speaker-data/*.zip')

if not os.path.exists(f'{DATASET_DIR}/development/images'):
    if data_zips:
        print(f'Extracting dataset from: {data_zips[0]} (~2 min)...')
        !unzip -q "{data_zips[0]}" -d "{DATASET_DIR}/"
        print('Done')
    else:
        print('ERROR: Dataset zip not found!')
        print('Add your manga-speaker-data dataset to this notebook!')
        !ls /kaggle/input/
else:
    print('Dataset already extracted')

imgs = len(glob.glob(f'{DATASET_DIR}/development/images/**/*.jpg', recursive=True))
print(f'Found {imgs} development images')

In [ ]:
# ── Cell 5: Verify data structure ─────────────────────────────────────────────
import json
from collections import Counter

with open('splits.json') as f:
    splits = json.load(f)
with open('dataset/development/labels.jsonl') as f:
    labels = [json.loads(l) for l in f]

train_ids = set(splits['train'])
print(f'Train: {len(splits["train"])} seqs | Val: {len(splits["val"])} seqs | Total labels: {len(labels)}')

spk_counter = Counter()
for seq in labels:
    if seq['sequence_id'] in train_ids:
        for page in seq['pages']:
            for line in page:
                spk_counter[line['speaker']] += 1

print('\nSpeaker distribution in training data:')
for s, c in spk_counter.most_common(10):
    print(f'  {s:20s} {c}')

In [ ]:
# ── Cell 6: TRAIN ─────────────────────────────────────────────────────────────
#
# Kaggle-specific notes:
#   - Output adapter saves to /kaggle/working/... (auto-downloadable)
#   - T4 x2: both GPUs used via device_map=auto
#   - P100: 16GB VRAM, can use rank=32 without 4bit
#   - We use --load-4bit to be safe (works on both T4 and P100)

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU(s):', result.stdout.strip())

# Check if P100 (16GB) - can skip 4bit
is_p100 = 'P100' in result.stdout
load_4bit_flag = '' if is_p100 else '--load-4bit'
print(f'Using 4bit quantization: {not is_p100}')

In [ ]:
# ── Cell 7: Run Training (~3-4 hours) ─────────────────────────────────────────

import subprocess, sys

cmd = [
    sys.executable, 'scripts/train_lora.py',
    '--dataset', 'dataset',
    '--labels', 'dataset/development/labels.jsonl',
    '--splits', 'splits.json',
    '--split', 'all',
    '--model-id', 'Qwen/Qwen2.5-VL-3B-Instruct',
    '--out', '/kaggle/working/adapters/qwen_manga_lora_v2',
    '--epochs', '2',
    '--lr', '2e-4',
    '--grad-accum', '4',
    '--rank', '32',
    '--context-pages', '1',
    '--max-pixels', '501760',
    '--seed', '42',
]

# Add --load-4bit only if on T4 (not P100)
if load_4bit_flag:
    cmd.append('--load-4bit')

print('Running:', ' '.join(cmd))
print('---')

# Stream output live
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit code: {proc.returncode}')


In [ ]:
# ── Cell 8: Verify adapter saved correctly ─────────────────────────────────────
import os, glob

adapter_files = glob.glob('/kaggle/working/adapters/qwen_manga_lora_v2/**', recursive=True)
print(f'Adapter files ({len(adapter_files)} total):')
for f in adapter_files[:10]:
    print(' ', f)

if len(adapter_files) > 3:
    print('\n✅ Adapter saved successfully!')
else:
    print('\n❌ Adapter missing — training may have failed')

In [ ]:
# ── Cell 9: Run inference on TEST set ─────────────────────────────────────────
import subprocess, sys

cmd = [
    sys.executable, 'scripts/run_inference.py',
    '--dataset', 'dataset',
    '--ids-from', 'dataset/sample_submission.jsonl',
    '--out', '/kaggle/working/test_predictions_v2.jsonl',
    '--config', 'configs/default.yaml',
    '--adapter', '/kaggle/working/adapters/qwen_manga_lora_v2',
    '--set', 'model.load_4bit=true',
           'model.model_id=Qwen/Qwen2.5-VL-3B-Instruct',
]

print('Running inference on test set...')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit code: {proc.returncode}')

In [ ]:
# ── Cell 10 (Optional): Score on validation set ────────────────────────────────
import subprocess, sys, json

# Run val inference
cmd_inf = [
    sys.executable, 'scripts/run_inference.py',
    '--dataset', 'dataset',
    '--ids-from', 'splits.json',
    '--split', 'val',
    '--out', '/kaggle/working/val_predictions_v2.jsonl',
    '--config', 'configs/default.yaml',
    '--adapter', '/kaggle/working/adapters/qwen_manga_lora_v2',
    '--set', 'model.load_4bit=true', 'model.model_id=Qwen/Qwen2.5-VL-3B-Instruct',
]
proc = subprocess.Popen(cmd_inf, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

# Score
cmd_sc = [
    sys.executable, 'dataset/score.py',
    '--predictions', '/kaggle/working/val_predictions_v2.jsonl',
    '--labels', 'dataset/development/labels.jsonl',
    '--output', '/kaggle/working/scores_v2.json',
    '--split', 'val',
    '--splits', 'splits.json',
]
subprocess.run(cmd_sc)

with open('/kaggle/working/scores_v2.json') as f:
    sc = json.load(f)
m = sc['macro']
print('\n=== v2 Validation Scores ===')
print(f"  balanced_joint_f1:   {m['balanced_joint_f1']:.4f}  (v1 was 0.1523)")
print(f"  speaker_accuracy:    {m['speaker_accuracy_on_matched']:.4f}  (v1 was 0.5704)")
print(f"  text_order_score:    {m['text_order_score']:.4f}  (v1 was 0.5865)")
print(f"  joint_f1:            {m['joint_f1']:.4f}  (v1 was 0.3761)")

In [ ]:
# ── Cell 11: Check output files (downloadable from Kaggle) ─────────────────────
import os, glob

output_files = glob.glob('/kaggle/working/**', recursive=False)
print('Files in /kaggle/working/ (all auto-downloadable):')
for f in output_files:
    size = os.path.getsize(f) if os.path.isfile(f) else '-'
    print(f'  {f}  {size}')

print()
print('To download:')
print('  1. Click "Save Version" (top right) → Save & Run All')
print('  2. Go to the Output tab')
print('  3. Download test_predictions_v2.jsonl')